# Moving annual median rent by suburb and quarter

**Goal:** reshape the DELWP/DFFH *Moving annual median rent by suburb and town* workbook into a tidy wide table where:

- **rows** = suburb
- **columns** = quarter (e.g. `Sep 2025`)
- **values** = median rent

`Group Total` rows and the region-name column are dropped — only individual suburbs are kept.

The source sheet has a two-row header (quarter label, then `Count` / `Median`) sitting above the suburb rows, and every quarter is stored as a `(Count, Median)` column pair. Rather than hard-coding column numbers, the code below locates the `Median` column for each quarter by reading the header itself, so it keeps working if columns are inserted, removed, or re-ordered.

## 1. Imports

In [14]:
from pathlib import Path

import numpy as np
import openpyxl
import pandas as pd


## 2. Configuration

Adjust `SHEET_NAME` to reshape a different property type (e.g. `"2 bedroom flat"`). All other settings describe the fixed layout of the source workbook and should not normally need to change.

In [15]:
INPUT_PATH = Path(
    "../../external_data/rent_by_qtr_suburb.xlsx"
)
OUTPUT_PATH = Path(
    "../../external_data/rent_by_qtr_suburb_refactored.csv"
)

SHEET_NAME = "All properties"

QUARTER_HEADER_ROW = 2
METRIC_HEADER_ROW = 3
DATA_START_ROW = 4

REGION_COL = 1
SUBURB_COL = 2

MEDIAN_LABEL = "Median"
GROUP_TOTAL_LABEL = "Group Total"

## 3. Locate the `Median` column for every quarter

For each column, `QUARTER_HEADER_ROW` holds the quarter label (e.g. `Sep 2025`) and `METRIC_HEADER_ROW` holds `Count` or `Median`. We only keep the `Median` columns, in the order they appear left to right.

In [16]:
def get_quarter_median_columns(worksheet):
    """Map each quarter label to its ``Median`` column index.

    Parameters
    ----------
    worksheet : openpyxl.worksheet.worksheet.Worksheet
        The sheet to inspect.

    Returns
    -------
    dict[str, int]
        ``{quarter_label: column_index}``, in sheet column order.
        ``column_index`` is 1-based, matching openpyxl's
        ``cell(row, column)`` convention.

    Raises
    ------
    ValueError
        If no ``Median`` columns are found at all, which would
        indicate the header layout has changed unexpectedly.
    """
    quarter_row = worksheet[QUARTER_HEADER_ROW]
    metric_row = worksheet[METRIC_HEADER_ROW]

    quarter_columns = {}
    for quarter_cell, metric_cell in zip(quarter_row, metric_row):
        metric_value = metric_cell.value
        quarter_value = quarter_cell.value

        if metric_value is None or quarter_value is None:
            continue

        is_median_col = str(metric_value).strip() == MEDIAN_LABEL
        if is_median_col:
            label = str(quarter_value).strip()
            quarter_columns[label] = quarter_cell.column

    if not quarter_columns:
        raise ValueError(
            "No 'Median' columns found — check QUARTER_HEADER_ROW "
            "and METRIC_HEADER_ROW against the source file."
        )
    return quarter_columns


## 4. Extract suburb-level median rents

Walks every data row, skips `Group Total` rows (and any blank rows), and reads the median rent out of each quarter column found above. Non-numeric placeholders such as `"-"` are treated as missing data (`NaN`) rather than dropped or coerced to zero.

In [17]:
def to_numeric_or_nan(value):
    """Coerce a cell value to ``float``, or ``np.nan`` if that
    fails (covers blanks, ``None``, and placeholders like
    ``"-"``).
    """
    try:
        return float(value)
    except (TypeError, ValueError):
        return np.nan


def extract_suburb_medians(worksheet, quarter_columns):
    """Build a suburb -> {quarter: median rent} mapping.

    Parameters
    ----------
    worksheet : openpyxl.worksheet.worksheet.Worksheet
        The sheet to read.
    quarter_columns : dict[str, int]
        Output of :func:`get_quarter_median_columns`.

    Returns
    -------
    dict[str, dict[str, float]]
        Keyed by suburb name; values are keyed by quarter label.

    Raises
    ------
    ValueError
        If the same suburb name appears more than once, since
        that would silently overwrite data in the final table.
    """
    suburb_medians = {}

    for row in worksheet.iter_rows(min_row=DATA_START_ROW):
        suburb_cell = row[SUBURB_COL - 1]
        suburb = suburb_cell.value

        if suburb is None:
            continue

        suburb = str(suburb).strip()
        if not suburb or suburb == GROUP_TOTAL_LABEL:
            continue

        if suburb in suburb_medians:
            raise ValueError(
                f"Duplicate suburb name encountered: {suburb!r}. "
                "Update the extraction logic to disambiguate "
                "(e.g. by prefixing the region) before proceeding."
            )

        row_number = suburb_cell.row
        medians = {}
        for quarter, col_idx in quarter_columns.items():
            raw_value = worksheet.cell(
                row=row_number, column=col_idx
            ).value
            medians[quarter] = to_numeric_or_nan(raw_value)

        suburb_medians[suburb] = medians

    return suburb_medians


## 5. Run the extraction and assemble the wide table

In [18]:
def build_median_rent_table(path, sheet_name):
    """Load ``path`` and return a wide suburb x quarter median
    rent table for ``sheet_name``.

    Parameters
    ----------
    path : pathlib.Path
        Path to the source ``.xlsx`` workbook.
    sheet_name : str
        Name of the sheet (property type) to reshape.

    Returns
    -------
    pandas.DataFrame
        Index is the suburb name (``Suburb``); columns are quarter
        labels in their original left-to-right sheet order;
        values are median rents.
    """
    if not path.exists():
        raise FileNotFoundError(f"Input workbook not found: {path}")

    workbook = openpyxl.load_workbook(path, data_only=True)
    if sheet_name not in workbook.sheetnames:
        raise KeyError(
            f"Sheet {sheet_name!r} not found. Available sheets: "
            f"{workbook.sheetnames}"
        )
    worksheet = workbook[sheet_name]

    quarter_columns = get_quarter_median_columns(worksheet)
    suburb_medians = extract_suburb_medians(worksheet, quarter_columns)

    table = pd.DataFrame.from_dict(suburb_medians, orient="index")
    table = table[list(quarter_columns.keys())]
    table.index.name = "Suburb"
    table.columns.name = "Quarter"
    return table.sort_index()


median_rent = build_median_rent_table(INPUT_PATH, SHEET_NAME)
median_rent.shape


(146, 103)

## 6. Sanity checks

Quick guards to catch a header-layout change or a bad parse before trusting the output.

In [19]:
assert median_rent.index.is_unique, "Suburb names are not unique."

no_group_totals = GROUP_TOTAL_LABEL not in median_rent.index
assert no_group_totals, "A 'Group Total' row leaked into the table."

has_numeric_dtypes = median_rent.dtypes.eq(float).all()
assert has_numeric_dtypes, "Non-numeric columns found in the table."

print(f"{median_rent.shape[0]} suburbs x {median_rent.shape[1]} "
      "quarters")


146 suburbs x 103 quarters


## 7. Preview and export

In [20]:
median_rent.iloc[:, -8:].head(10)

Quarter,Dec 2023,Mar 2024,Jun 2024,Sep 2024,Dec 2024,Mar 2025,Jun 2025,Sep 2025
Suburb,,,,,,,,
Albert Park-Middle Park-West St Kilda,600.0,660.0,675.0,693.0,700.0,700.0,695.0,698.0
Altona,490.0,510.0,530.0,550.0,550.0,550.0,550.0,550.0
Armadale,560.0,560.0,583.0,600.0,600.0,625.0,620.0,625.0
Aspendale-Chelsea-Carrum,550.0,575.0,590.0,600.0,600.0,600.0,615.0,620.0
Bairnsdale,420.0,420.0,425.0,430.0,450.0,450.0,460.0,460.0
Ballarat,380.0,380.0,390.0,390.0,395.0,395.0,400.0,400.0
Balwyn,650.0,680.0,680.0,700.0,700.0,700.0,720.0,740.0
Bayswater,485.0,495.0,500.0,520.0,550.0,550.0,550.0,550.0
Belmont-Grovedale,470.0,475.0,480.0,480.0,490.0,490.0,495.0,500.0


In [21]:
median_rent.to_csv(OUTPUT_PATH)